# Test all_splits Function

In [ ]:
# %%
import pandas as pd
import numpy as np
from pathlib import Path

from FIT_python.config import RAW_DIR
from FIT_python.splits_wrapper import all_splits
from FIT_python.split_utils import (
    group_stratified_kfold,
    create_train_test_split_otter
)

pd.set_option('display.max_columns', None)

# %% [markdown]
## 1. Übersicht der Datensätze

# %%
raw_dir = Path(RAW_DIR)
print(f"Raw data directory: {raw_dir}")
# Liste alle CSV- und Parquet-Dateien
file_paths = [p for p in raw_dir.iterdir() if p.suffix.lower() in ['.csv', '.parquet']]
print("Verfügbare Datendateien:")
for p in file_paths:
    print(f" - {p.name}")
print()

# Hilfsfunktion zum Laden
# Name aus splits keys kann z.B. 'Amur_Tiger' sein: wir normalisieren beides für Abgleich.
def load_full_df(name: str) -> pd.DataFrame:
    normalized = name.replace('_', ' ').lower()
    for p in file_paths:
        if normalized in p.stem.lower():
            print(f"Lade kompletten DataFrame für {name} von {p.name}")
            if p.suffix.lower() == '.csv':
                return pd.read_csv(p)
            else:
                return pd.read_parquet(p)
    print(f"Kein komplettes DataFrame gefunden für {name}")
    return None

# %% [markdown]
## 2. Test: all_splits-Funktion

# %%
splits = all_splits(raw_dir)
print(f"all_splits liefert {len(splits)} Datensätze")
assert isinstance(splits, dict), "all_splits sollte ein dict zurückgeben"

for name, parts in splits.items():
    print(f"\n=== Datensatz: {name} ===")
    # prüfe keys
    print(f"Keys: {list(parts.keys())}")
    assert 'train' in parts and 'test' in parts, "Fehlende Keys 'train'/'test'"
    train_df = parts['train']
    test_df = parts['test']
    print(f"Train-Größe: {len(train_df)}, Test-Größe: {len(test_df)}")
    # Show sample rows
    display(train_df.head(2))
    display(test_df.head(2))
    # vollständiges DF prüfen
    df_all = load_full_df(name)
    if df_all is not None:
        total = len(df_all)
        sum_splits = len(train_df) + len(test_df)
        print(f"Total: {total}, sum(train+test): {sum_splits}")
        assert total == sum_splits, (
            f"Anzahl Zeilen stimmt nicht überein für {name}: {total} vs {sum_splits}"
        )
    # prüfe Gruppenüberschneidung
    train_ids = set(train_df['individual_id'])
    test_ids = set(test_df['individual_id'])
    overlap = train_ids & test_ids
    print(f"Anzahl überlappender individual_id: {len(overlap)}")
    assert not overlap, (
        f"Train und Test teilen sich individual_id(s) in {name}: {overlap}"
    )
    print(f"✔ Train/Test-Check erfolgreich für {name}")



In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold

from FIT_python.config import RAW_DIR
from FIT_python.splits_wrapper import all_splits

NUM_FOLDS = 3
GLOBAL_RANDOM_SEED = 42
GROUP_COL = "individual_id"

def group_stratified_kfold(
    df: pd.DataFrame,
    n_splits: int = NUM_FOLDS,
    random_state: int = GLOBAL_RANDOM_SEED,
    group_col: str = GROUP_COL,
    stratify_col: str = "sex"
) -> pd.DataFrame:
    """
    Assigns a 'Fold' column via
      - StratifiedGroupKFold on groups, stratified by stratify_col,
        if every class has at least n_splits groups.
      - Otherwise GroupKFold on groups only.
    """
    # Build DataFrame of unique groups and their class
    individuals = (
        df[[group_col, stratify_col]]
        .drop_duplicates(subset=[group_col])
        .fillna({stratify_col: "unknown"})
    )
    class_counts = individuals[stratify_col].value_counts()
    use_strat = (class_counts >= n_splits).all()

    if use_strat:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_fn = lambda: splitter.split(individuals, individuals[stratify_col], groups=individuals[group_col])
    else:
        splitter = GroupKFold(n_splits=n_splits)
        split_fn = lambda: splitter.split(individuals, groups=individuals[group_col])

    fold_series = pd.Series(-1, index=df.index, name="Fold")
    for fold, (_, val_idx) in enumerate(split_fn()):
        fold_groups = individuals.iloc[val_idx][group_col]
        fold_series.loc[df[group_col].isin(fold_groups)] = fold

    return df.assign(Fold=fold_series)


# Generate splits and summarize by fold
raw_dir = Path(RAW_DIR)
splits = all_splits(raw_dir)

for name, parts in splits.items():
    print(f"=== Dataset: {name} ===")
    df_train = parts['train'].copy()

    # Ensure 'sex' column exists
    if 'sex' not in df_train.columns:
        df_train['sex'] = 'unknown'

    # Create debug stratify column if missing
    if 'proportion' not in df_train.columns:
        ids = pd.to_numeric(df_train['individual_id'], errors='coerce').fillna(0)
        df_train['proportion'] = np.floor(ids % NUM_FOLDS)

    folded = group_stratified_kfold(df_train, n_splits=NUM_FOLDS)

    # Summary per individual
    summary = (
        folded
        .groupby(['individual_id', 'sex', 'Fold'])
        .size()
        .reset_index(name='row_count')
    )
    print("Summary (individual_id, sex, Fold, row_count):")
    display(summary)

    # Unique individual counts per fold and sex
    counts = (
        folded
        .drop_duplicates(['individual_id', 'sex', 'Fold'])
        .groupby(['Fold', 'sex'])['individual_id']
        .nunique()
        .unstack(fill_value=0)
    )
    print("Unique individuals per Fold and sex:")
    display(counts)
    print()
